## Transform Drivers Data
1. Read bronze drivers table
2. Keep only the columns required for analytics(Drop url Column)
3. Standardise column names using snake_case (driverId -> driver_id, dateOfbirth -> date_of_birth)
4. Concatenate name.givenName and name.familyname to create a new column called driver_name and transform the value to Title Case
5. Remove duplicate records
6. Transform values of columns nationality to Title Case
7. Write the transformed data to silver drivers table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

#### 1. Read bronze circuits table

In [0]:
drivers_df = spark.read.table(bronze_table).filter((F.col("batch_id") == v_batch_id))

#### 2. Keep only the columns required for analytics(Drop url Column)

In [0]:
from pyspark.sql import functions as F
drivers_selected_df = drivers_df.drop(F.col("url"))

#### Step 3 & 4 - Standardise column names
 - Standardise column names using snake_case (driverId -> driver_id,dateOfbirth -> date_of_birth)
 - Concatenate name.givenName and name.familyname to create a new column called driver_name and transform the value to Title Case


In [0]:
drivers_renamed_df = (
    drivers_selected_df
    .withColumnsRenamed({
        "driverId" : "driver_id",
        "dateOfbirth" : "date_of_birth"
    })
    .withColumn("driver_name", F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName")))
    .drop(F.col("name"))
)


#### Step 5 - Remove duplicate records

In [0]:
drivers_distinct_df = drivers_renamed_df.dropDuplicates(["driver_id"])

#### Step 6 - Transform values of columns nationality to Title Case

In [0]:
drivers_final_df = (
    drivers_distinct_df
    .withColumn("nationality",F.initcap(F.col("nationality")))
)

#### Step 7 -  Write the transformed data to silver circuits table

In [0]:
merge_condition="tgt.driver_id = src.driver_id"
columns_to_update=["driver_name","date_of_birth","nationality","ingestion_timestamp","source_file","batch_id"]
write_to_silver(drivers_final_df, silver_table, merge_condition, columns_to_update)
